# 02 · multilingual-e5-small embeddings for a diverse S1 / S2 / S3 sample

**Input:** the per-country parquet files written by `01_eda_preprocessing.ipynb` into `processed/`
(`train_s1_US.parquet`, `test_s1_France.parquet`, `train_s2_India.parquet`, …), plus
`train_ground_truth.tsv` from the competition data (used only to link the sampled records).

**Output** (`embeddings/multilingual-e5-small/`):

| File | Content |
|---|---|
| `s1_embeddings.npy`, `s2_embeddings.npy`, `s3_embeddings.npy` | float16, one unit-norm 384-d row per sampled record |
| `s1_meta.parquet`, `s2_meta.parquet`, `s3_meta.parquet` | row-aligned metadata: `entity_id`, split, country, `kind`, true S1 link, diversity tags, embedded text |
| `manifest.json` | settings, counts, tag shares (sample vs population), timing |
| `faiss/s1_buckets.parquet` | per S1: its top-`TOP_K` S2/S3 candidates (the "bucket") with rank, retrieval score and original e5 cosine (`BUCKET_METHOD`, IVF-PQ by default) |
| `faiss/s1_buckets_exact.parquet`, `faiss/s1_buckets_ivfpq.parquet` | the buckets from each search method (the IVF-PQ file only when `BUCKET_METHOD=ivfpq`) |
| `faiss/mean_<country>.npy`, `faiss/index_<country>.faiss` | the mean used to center that country's embeddings, and the IVF-PQ index over its centered S2+S3 embeddings (only when `BUCKET_METHOD=ivfpq`) |
| `faiss/bucket_recall.csv` | share of true matches inside the buckets, IVF-PQ vs exact search |
| `faiss/metric_f05.csv` | the challenge metric (macro F0.5 per S1) for the buckets: ceiling, baselines, tuned cosine cutoff |

**How the sample is made diverse** (`N_PER_SOURCE` records per source, default 100,000):
* **Countries:** each source's quota is split evenly across the countries it has. France exists
  only in test S1, so it appears in S1 only, unless `test_s2_*` / `test_s3_*` files are added
  to `processed/` (they are picked up automatically).
* **Linked groups:** S1 records of labeled countries come from train. For S2/S3, 70% are records
  whose true S1 is in the S1 sample (so true pairs exist inside the sample), 30% are no-match
  records. Any shortfall is filled with records owned by an S1 outside the sample.
* **Hard cases:** within every (source, country) quota, each diversity tag gets at least
  `MIN_TAG_SHARE` (5%) when available: native-script name, web/handle name, empty address,
  honorific, landmark address, postcode present, no legal form, one-word name. The rest is random.

**Text given to e5** (`TEXT_MODE`):
* `processed` (default): `name_norm | addr_norm` from `processed/`
* `latin`: `name_lat legal_form | addr_norm` (ASCII transliteration)
* `raw`: `business_name | business_address`

The `processed` step strips combining marks, which also removes some marks inside Indian scripts
(e.g. the virama ्), so native-script names reach e5 slightly altered. In our earlier benchmark
e5 read raw native script well (94.7% recall@5), so compare `raw` against `processed` on the
`tag_native_script` slice in the check at the end.

**Run on Kaggle:**
1. Attach the competition dataset and the `processed/` output of notebook 01. Add the 01
   notebook's output as an input, or run 01 in this session.
2. Accelerator: GPU T4 x2; both GPUs are used automatically for encoding and search. Internet on,
   to download `intfloat/multilingual-e5-small` (MIT); or attach the model and set `E5_MODEL` to its folder.
3. Run all. Encoding 300K short texts takes a few minutes.

**Bigger sample** (e.g. 500K per source): in the first code cell set `N_PER_SOURCE = 500_000` and
`BUCKET_METHOD = "exact"`. The second skips the IVF-PQ comparison, which is the slow CPU part at
that size. This needs 01's `FULL_RUN` output, since the sample-mode files hold only 40K rows each.

Settings live in the first code cell and can also be set as environment variables: `PROCESSED_DIR`,
`DATA_ROOT`, `EMB_DIR`, `N_PER_SOURCE`, `TEXT_MODE`, `E5_MODEL`, `TOP_K`, `NPROBE_FRAC`, `BUCKET_METHOD`.

The last part retrieves, for every sampled S1, its top-k S2/S3 records per country as that S1's
candidate bucket: exact search (on all GPUs when there are any), and a **FAISS IVF-PQ** index for
comparison. `faiss-cpu` (MIT) is installed if missing.

In [ ]:
import csv
import glob
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch

N_PER_SOURCE = int(os.environ.get("N_PER_SOURCE", 100_000))   # records per source (S1, S2, S3)
SEED = 42
LINKED_SHARE = 0.70          # S2/S3: share whose true S1 is in the S1 sample
MIN_TAG_SHARE = 0.05         # minimum share per diversity tag within each (source, country) quota
TEXT_MODE = os.environ.get("TEXT_MODE", "processed")           # processed | latin | raw
MODEL_NAME = os.environ.get("E5_MODEL", "intfloat/multilingual-e5-small")
PREFIX = "query: "           # E5 expects "query: " for symmetric similarity
MAX_SEQ_LENGTH = 128         # tokens; names + addresses are short
TOP_K = int(os.environ.get("TOP_K", 50))                      # bucket size per S1
BUCKET_METHOD = os.environ.get("BUCKET_METHOD", "ivfpq")       # ivfpq (IVF-PQ + exact, compared) | exact (exact only)
NPROBE_FRAC = float(os.environ.get("NPROBE_FRAC", 0.25))       # share of IVF clusters scanned per query
assert TEXT_MODE in {"processed", "latin", "raw"}, TEXT_MODE
assert BUCKET_METHOD in {"ivfpq", "exact"}, BUCKET_METHOD


def find_processed_dir():
    """Folder with 01's per-country parquet: $PROCESSED_DIR, ./processed, /kaggle/working, any Kaggle input."""
    candidates = [Path(os.environ["PROCESSED_DIR"])] if os.environ.get("PROCESSED_DIR") else []
    candidates += [Path.cwd() / "processed", Path("/kaggle/working/processed")]
    candidates += sorted({Path(p).parent for p in glob.glob("/kaggle/input/**/train_s1_*.parquet", recursive=True)})
    for c in candidates:
        if c.is_dir() and any(c.glob("train_s1_*.parquet")):
            return c
    raise FileNotFoundError("processed/ not found: run 01_eda_preprocessing.ipynb or set PROCESSED_DIR")


def find_ground_truth():
    """train_ground_truth.tsv under $DATA_ROOT, /kaggle/input, or the local dataset folders used by 01."""
    roots = [os.environ.get("DATA_ROOT", ""), "/kaggle/input"]
    here = Path.cwd().resolve()
    roots += [str(here.parent / "6ab10eb3b23ba_student_resource" / "student_resource" / "dataset"),
              str(here.parent / "Data" / "student_resource" / "dataset")]
    for root in roots:
        if root and os.path.isdir(root):
            hits = [h for h in glob.glob(os.path.join(root, "**", "train_ground_truth.tsv"), recursive=True)
                    if "__MACOSX" not in h]
            if hits:
                return Path(sorted(hits, key=len)[0])
    raise FileNotFoundError("train_ground_truth.tsv not found; set DATA_ROOT")


PROCESSED_DIR = find_processed_dir()
GT_FILE = find_ground_truth()
OUT_DIR = Path(os.environ.get("EMB_DIR") or ("/kaggle/working/embeddings" if os.path.isdir("/kaggle/working")
                                             else "embeddings")) / MODEL_NAME.rstrip("/").split("/")[-1]
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPUS = [f"cuda:{i}" for i in range(torch.cuda.device_count())] if DEVICE == "cuda" else []   # all used (T4 x2 on Kaggle)
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
# fp16 speeds up tensor-core GPUs (T4, A100) but is ~4x slower on GTX 16xx cards (measured on a GTX 1650).
USE_FP16 = DEVICE == "cuda" and "GTX 16" not in GPU_NAME
BATCH_SIZE = 1024 if USE_FP16 else 512                         # texts per GPU batch when encoding
print(f"processed: {PROCESSED_DIR}\nground truth: {GT_FILE}\noutput: {OUT_DIR}")
print(f"device: {len(GPUS)} x {GPU_NAME} | fp16: {USE_FP16} | batch: {BATCH_SIZE} | text mode: {TEXT_MODE} | "
      f"{N_PER_SOURCE:,} per source")

## Challenge metric: macro F0.5 per S1
From the problem statement: for every S1, F0.5 = 1.25·P·R / (0.25·P + R) over its predicted vs
true S2/S3 IDs, then averaged over **all** S1 (macro). An S1 with no true match scores 1.0 for an
empty prediction and 0.0 for any prediction. An S1 with matches scores 0.0 for an empty prediction.
The assert reproduces the worked example in the problem statement (0.714).

In [ ]:
def f05(predicted, true):
    """Challenge F0.5 for one S1: predicted and true are collections of S2/S3 entity IDs."""
    predicted, true = set(predicted), set(true)
    if not true:
        return 1.0 if not predicted else 0.0
    tp = len(predicted & true)
    if tp == 0:
        return 0.0
    precision, recall = tp / len(predicted), tp / len(true)
    return 1.25 * precision * recall / (0.25 * precision + recall)


def macro_f05(pred_by_s1, true_by_s1, s1_ids):
    """Mean challenge F0.5 over s1_ids (S1 missing from pred_by_s1 = empty prediction)."""
    return float(np.mean([f05(pred_by_s1.get(s, ()), true_by_s1.get(s, ())) for s in s1_ids]))


assert round(f05(["S2-00047", "S2-00193", "S3-00812"], ["S2-00047", "S3-00812"]), 3) == 0.714
assert f05([], []) == 1.0 and f05(["S2-1"], []) == 0.0 and f05([], ["S2-1"]) == 0.0
print("challenge metric OK (problem-statement example = 0.714)")

## Inventory of `processed/`

In [ ]:
FILE_RE = re.compile(r"^(train|test)_(s[123])_(.+)\.parquet$")
inventory = []
for f in sorted(PROCESSED_DIR.glob("*.parquet")):
    m = FILE_RE.match(f.name)
    if m:
        inventory.append({"split": m.group(1), "source": m.group(2), "country": m.group(3), "path": f,
                          "rows": pq.ParquetFile(f).metadata.num_rows})
INVENTORY = pd.DataFrame(inventory)
assert len(INVENTORY), f"no <split>_<source>_<country>.parquet files in {PROCESSED_DIR}"
if (INVENTORY["rows"] <= 40_000).all():
    print("NOTE: every file has <= 40,000 rows, so processed/ looks like 01's sample mode (first 40K rows "
          "per country). Few true S1-S2/S3 pairs will exist. Run 01 with FULL_RUN=true for linked groups.")
INVENTORY.drop(columns="path")

## Loading and diversity tags

In [ ]:
READ_COLS = ["entity_id", "business_name", "business_address", "country", "name_norm", "core_name",
             "legal_form", "domain_stem", "name_lat", "has_honorific", "addr_norm", "empty_addr",
             "missing_pin", "landmark_flag"]
STR_COLS = ["entity_id", "business_name", "business_address", "country", "name_norm", "core_name",
            "legal_form", "domain_stem", "name_lat", "addr_norm"]
INT_COLS = ["has_honorific", "empty_addr", "missing_pin", "landmark_flag"]
# Anything outside Latin, Latin-1/Extended and general punctuation counts as a native script.
RE_NATIVE = re.compile(r"[^\x00-\x7f\u00a0-\u024f\u1e00-\u1eff\u2000-\u206f\u20a0-\u20cf]")
RE_HANDLE = re.compile(r"^\s*[@#]\S+\s*$")
TAGS = ["tag_native_script", "tag_web_name", "tag_empty_address", "tag_honorific", "tag_landmark",
        "tag_has_postcode", "tag_no_legal_form", "tag_one_word_name"]


def read_processed(path):
    """Read one processed parquet with the needed columns; missing strings become '' and flags 0."""
    available = set(pq.read_schema(path).names)
    df = pd.read_parquet(path, columns=[c for c in READ_COLS if c in available])
    for c in STR_COLS:
        df[c] = df[c].fillna("").astype(str) if c in df else ""
    for c in INT_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int) if c in df else 0
    return df


def add_tags(df):
    """Boolean diversity tags computed from the processed columns."""
    names = df["business_name"].tolist()
    return df.assign(
        tag_native_script=[RE_NATIVE.search(x) is not None for x in names],
        tag_web_name=(df["domain_stem"] != "").to_numpy() | np.array([RE_HANDLE.match(x) is not None for x in names]),
        tag_empty_address=(df["empty_addr"] == 1) | (df["addr_norm"] == ""),
        tag_honorific=df["has_honorific"] == 1,
        tag_landmark=df["landmark_flag"] == 1,
        tag_has_postcode=df["missing_pin"] == 0,
        tag_no_legal_form=df["legal_form"] == "",
        tag_one_word_name=[len(x.split()) <= 1 for x in df["core_name"].tolist()])


def diverse_sample(df, n, rng):
    """Row labels of n rows: at least MIN_TAG_SHARE * n rows of every tag (when available), rest random."""
    if n <= 0 or len(df) == 0:
        return df.index[:0]
    if len(df) <= n:
        return df.index
    chosen = np.zeros(len(df), bool)
    per_tag = int(MIN_TAG_SHARE * n)
    for tag in TAGS:
        pool = np.flatnonzero(df[tag].to_numpy() & ~chosen)
        take = min(per_tag, len(pool))
        if take:
            chosen[rng.choice(pool, take, replace=False)] = True
    rest = np.flatnonzero(~chosen)
    need = n - int(chosen.sum())
    if need > 0:
        chosen[rng.choice(rest, min(need, len(rest)), replace=False)] = True
    return df.index[chosen]


def split_quota(total, keys):
    """Split an integer total as evenly as possible over keys."""
    base, extra = divmod(total, len(keys))
    return {k: base + (i < extra) for i, k in enumerate(keys)}


def tag_shares(df):
    """Percentage of rows carrying each tag."""
    return {t: round(100 * float(df[t].mean()), 2) for t in TAGS} if len(df) else {}

## Ground truth (links only)
Used to link sampled S2/S3 records to their S1 and to find no-match records. It is not used to build the text.

In [ ]:
gt = pd.read_csv(GT_FILE, sep="\t", dtype=object, keep_default_na=False, na_filter=False,
                 quoting=csv.QUOTE_NONE)
n_matches_of = gt.set_index("source1_entity_id")["matched_entity_ids"].map(
    lambda s: len(s.split(",")) if s else 0)
gt = gt[gt["matched_entity_ids"] != ""]
pairs = gt.assign(entity_id=gt["matched_entity_ids"].str.split(",")).explode("entity_id")
OWNER = pd.Series(pairs["source1_entity_id"].to_numpy(), index=pairs["entity_id"].to_numpy())
del gt, pairs
print(f"{len(OWNER):,} S2/S3 records have a true S1; {int((n_matches_of == 0).sum()):,} train S1 are singletons")

## Sample S1
Labeled countries are drawn from train (so their S2/S3 matches can be linked). Countries that
exist only in test (France) are drawn from test.

In [ ]:
rng = np.random.default_rng(SEED)
POPULATION_TAGS = {}


def files_for(source, country):
    """Processed files of one source and country, train first."""
    sel = INVENTORY[(INVENTORY["source"] == source) & (INVENTORY["country"] == country)]
    return sel.sort_values("split", key=lambda s: s.map({"train": 0, "test": 1}))


s1_parts = []
s1_countries = sorted(INVENTORY.loc[INVENTORY["source"] == "s1", "country"].unique())
for country, quota in split_quota(N_PER_SOURCE, s1_countries).items():
    for _, f in files_for("s1", country).iterrows():
        if quota <= 0:
            break
        df = add_tags(read_processed(f["path"]))
        POPULATION_TAGS[f"s1/{f['split']}/{country}"] = tag_shares(df)
        picked = df.loc[diverse_sample(df, quota, rng)].assign(split=f["split"])
        s1_parts.append(picked)
        quota -= len(picked)
        del df
S1 = pd.concat(s1_parts, ignore_index=True)
S1["n_true_matches"] = S1["entity_id"].map(n_matches_of).where(S1["split"] == "train")
S1["kind"] = np.where(S1["split"] != "train", "unlabeled",
                      np.where(S1["n_true_matches"] > 0, "has_matches", "singleton"))
SAMPLED_S1 = set(S1["entity_id"])
S1.groupby(["country", "split", "kind"]).size().rename("records").to_frame()

## Sample S2 and S3
Per country: 70% records linked to a sampled S1, 30% no-match records, each chosen with the tag
minimums. Shortfalls are filled with records whose S1 is outside the sample, then with anything
left. Countries that exist only in test contribute unlabeled records.

In [ ]:
def sample_source(source):
    """Diverse, partly linked sample of one of S2/S3."""
    parts = []
    countries = sorted(INVENTORY.loc[INVENTORY["source"] == source, "country"].unique())
    for country, quota in split_quota(N_PER_SOURCE, countries).items():
        frames = []
        for _, f in files_for(source, country).iterrows():
            df = add_tags(read_processed(f["path"])).assign(split=f["split"])
            POPULATION_TAGS[f"{source}/{f['split']}/{country}"] = tag_shares(df)
            frames.append(df)
        pool = pd.concat(frames, ignore_index=True)
        del frames
        owner = pool["entity_id"].map(OWNER).where(pool["split"] == "train")
        pool["true_s1"] = owner.fillna("")
        pool["kind"] = np.select(
            [pool["split"] != "train", owner.isna(), owner.isin(SAMPLED_S1)],
            ["unlabeled", "no_match", "linked"], default="other_owner")
        plan = [("linked", round(LINKED_SHARE * quota)), ("no_match", quota - round(LINKED_SHARE * quota))]
        picked = []
        for kind, n in plan:
            idx = diverse_sample(pool[pool["kind"] == kind], n, rng)
            picked.append(idx)
        taken = pool.index.isin(np.concatenate([np.asarray(i) for i in picked]))
        for kind in ("other_owner", "linked", "no_match", "unlabeled"):   # fill any shortfall
            short = quota - int(taken.sum())
            if short <= 0:
                break
            idx = diverse_sample(pool[(pool["kind"] == kind) & ~taken], short, rng)
            taken |= pool.index.isin(idx)
        parts.append(pool[taken])
        n_linked = int((pool[taken]["kind"] == "linked").sum())
        print(f"{source} {country}: {int(taken.sum()):,} records ({n_linked:,} linked to a sampled S1)")
        del pool
    return pd.concat(parts, ignore_index=True)


S2 = sample_source("s2")
S3 = sample_source("s3")
pd.concat({"s2": S2.groupby(["country", "kind"]).size(), "s3": S3.groupby(["country", "kind"]).size()},
          axis=1).fillna(0).astype(int)

### Diversity check: tag shares in the sample vs the full files

In [ ]:
rows = []
for source, df in (("s1", S1), ("s2", S2), ("s3", S3)):
    for country, g in df.groupby("country"):
        rows.append({"source": source, "country": country, "records": len(g), **tag_shares(g)})
SAMPLE_TAGS = pd.DataFrame(rows).set_index(["source", "country"])
print("sample (% of records with each tag):")
display(SAMPLE_TAGS)
print("full processed files (% of records with each tag):")
pd.DataFrame(POPULATION_TAGS).T

## Embed with multilingual-e5-small
GPU use:
* Each unique text is encoded once, in fp16 on tensor-core GPUs (T4), `BATCH_SIZE` texts per batch.
* Texts are sorted by length first, so each batch holds texts of similar length and little padding.
* With several GPUs (Kaggle T4 x2), one encoding process runs per GPU. Chunks of `CHUNK_SIZE`
  texts are handed out to whichever GPU is free, and the processes stop once all sources are embedded.

In [ ]:
import inspect

from sentence_transformers import SentenceTransformer

CHUNK_SIZE = 16 * BATCH_SIZE     # texts per work unit handed to one GPU process


def build_text(df):
    """Text given to e5, according to TEXT_MODE."""
    if TEXT_MODE == "raw":
        names, addrs = df["business_name"], df["business_address"]
    elif TEXT_MODE == "latin":
        names, addrs = (df["name_lat"] + " " + df["legal_form"]).str.strip(), df["addr_norm"]
    else:
        names, addrs = df["name_norm"], df["addr_norm"]
    return [f"{n.strip()} | {a.strip()}" if a.strip() else n.strip() for n, a in zip(names, addrs)]


REUSE_EMBEDDINGS = True      # load saved embeddings when the same records, texts and model were embedded before
_model, _pool = None, None


def get_model():
    """Load the e5 model once, on first use (kept on the CPU when it is copied to several GPUs)."""
    global _model
    if _model is None:
        _model = SentenceTransformer(MODEL_NAME, device="cpu" if len(GPUS) > 1 else DEVICE)
        _model.max_seq_length = MAX_SEQ_LENGTH
        if USE_FP16:
            _model.half()
    return _model


def get_pool():
    """One encoding process per GPU, started once and reused for S1, S2 and S3."""
    global _pool
    if _pool is None:
        _pool = get_model().start_multi_process_pool(target_devices=GPUS)
    return _pool


def stop_pool():
    """Stop the encoding processes (frees their GPU memory for the search below)."""
    global _pool
    if _pool is not None:
        SentenceTransformer.stop_multi_process_pool(_pool)
        _pool = None


def embed(texts):
    """Unit-norm float16 embeddings aligned with texts, and seconds taken.

    Each unique text is encoded once, longest first, so batches hold texts of similar length.
    With several GPUs the chunks are shared among one process per GPU.
    """
    uniq, inverse = np.unique(np.asarray(texts, dtype=object), return_inverse=True)
    order = np.argsort([-len(t) for t in uniq], kind="stable")
    batch = [PREFIX + t for t in uniq[order]]
    t0 = time.time()
    model = get_model()
    if len(GPUS) > 1:
        if "pool" in inspect.signature(model.encode).parameters:          # sentence-transformers >= 5
            vecs = model.encode(batch, pool=get_pool(), batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE,
                                convert_to_numpy=True, show_progress_bar=True)
        else:
            vecs = model.encode_multi_process(batch, get_pool(), batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE)
    else:
        vecs = model.encode(batch, batch_size=BATCH_SIZE, convert_to_numpy=True, show_progress_bar=True)
    vecs = np.asarray(vecs, dtype=np.float32)
    vecs /= np.maximum(np.linalg.norm(vecs, axis=1, keepdims=True), 1e-12)
    in_uniq_order = np.empty_like(vecs)
    in_uniq_order[order] = vecs                                            # undo the length sort
    return in_uniq_order[inverse].astype(np.float16), time.time() - t0


def saved_embeddings(source, df):
    """Previously saved embeddings for exactly these records and texts with the same model settings, else None."""
    meta_path, emb_path, man_path = (OUT_DIR / f"{source}_meta.parquet", OUT_DIR / f"{source}_embeddings.npy",
                                     OUT_DIR / "manifest.json")
    if not (REUSE_EMBEDDINGS and meta_path.exists() and emb_path.exists() and man_path.exists()):
        return None
    old = json.loads(man_path.read_text(encoding="utf-8"))
    if (old.get("model"), old.get("prefix"), old.get("max_seq_length")) != (MODEL_NAME, PREFIX, MAX_SEQ_LENGTH):
        return None
    meta = pd.read_parquet(meta_path, columns=["entity_id", "text"])
    if meta["entity_id"].tolist() != df["entity_id"].tolist() or meta["text"].tolist() != df["text"].tolist():
        return None
    return np.load(emb_path), old.get("timing", {}).get(source)


EMB, TIMING = {}, {}
META_COLS = ["entity_id", "split", "country", "kind", "business_name", "business_address", "name_norm",
             "addr_norm", "text"] + TAGS
for source, df in (("s1", S1), ("s2", S2), ("s3", S3)):
    df["text"] = build_text(df)
    cached = saved_embeddings(source, df)
    if cached is not None:
        EMB[source], TIMING[source] = cached
        print(f"{source}: {EMB[source].shape} loaded from {OUT_DIR} (same records, texts and model)")
        continue
    EMB[source], secs = embed(df["text"].tolist())
    TIMING[source] = {"records": len(df), "seconds": round(secs, 1), "texts_per_sec": round(len(df) / max(secs, 1e-9))}
    extra = ["n_true_matches"] if source == "s1" else ["true_s1"]
    np.save(OUT_DIR / f"{source}_embeddings.npy", EMB[source])
    df[META_COLS + extra].to_parquet(OUT_DIR / f"{source}_meta.parquet", index=False)
    print(f"{source}: {EMB[source].shape} saved in {secs:.0f}s ({TIMING[source]['texts_per_sec']:,} texts/s)")
stop_pool()

manifest = {
    "model": MODEL_NAME, "prefix": PREFIX, "text_mode": TEXT_MODE, "max_seq_length": MAX_SEQ_LENGTH,
    "device": GPU_NAME, "gpus": len(GPUS), "fp16": USE_FP16, "batch_size": BATCH_SIZE, "seed": SEED,
    "n_per_source": N_PER_SOURCE,
    "linked_share": LINKED_SHARE, "min_tag_share": MIN_TAG_SHARE, "processed_dir": str(PROCESSED_DIR),
    "counts": {s: {f"{c}/{k}": int(v) for (c, k), v in d.groupby(["country", "kind"]).size().items()}
               for s, d in (("s1", S1), ("s2", S2), ("s3", S3))},
    "sample_tag_shares": {f"{s}/{c}": r for (s, c), r in SAMPLE_TAGS.to_dict("index").items()},
    "population_tag_shares": POPULATION_TAGS, "timing": TIMING,
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=1, default=str), encoding="utf-8")
print("saved to", OUT_DIR)

## Quick check: do the embeddings find the true S1?
For every linked S2/S3 record: its cosine to its own S1, its rank among all sampled S1 of the same
country, and the best wrong S1. For no-match records: their best S1 cosine. The numbers are
optimistic, because the S1 index here holds only the sample, not all S1 records.

In [ ]:
def check(source, df):
    """Recall@k and cosine summaries of one source against the sampled S1 of the same country."""
    rows = []
    for country in sorted(df["country"].unique()):
        s_rows = np.flatnonzero((S1["country"] == country).to_numpy())
        q_rows = np.flatnonzero(((df["country"] == country) & df["kind"].isin(["linked", "no_match"])).to_numpy())
        if not len(s_rows) or not len(q_rows):
            continue
        col_of = {e: j for j, e in enumerate(S1["entity_id"].to_numpy()[s_rows])}
        true_col = np.array([col_of.get(t, -1) for t in df["true_s1"].to_numpy()[q_rows]])
        index = torch.from_numpy(EMB["s1"][s_rows]).to(DEVICE).float()
        rank = np.full(len(q_rows), -1)
        true_cos = np.full(len(q_rows), np.nan, np.float32)
        best_cos = np.full(len(q_rows), np.nan, np.float32)
        for s in range(0, len(q_rows), 2048):
            sims = torch.from_numpy(EMB[source][q_rows[s:s + 2048]]).to(DEVICE).float() @ index.T
            best_cos[s:s + 2048] = sims.max(dim=1).values.cpu().numpy()
            t = torch.from_numpy(true_col[s:s + 2048]).to(DEVICE)
            has = torch.nonzero(t >= 0).flatten()
            if len(has):
                tc = sims[has, t[has]]
                true_cos[s + has.cpu().numpy()] = tc.cpu().numpy()
                rank[s + has.cpu().numpy()] = (sims[has] > tc[:, None]).sum(dim=1).cpu().numpy()
        sub = df.iloc[q_rows]
        linked = true_col >= 0
        slices = {"all linked": linked}
        slices.update({t: linked & sub[t].to_numpy() for t in TAGS})
        for name, m in slices.items():
            if m.sum() < 20:
                continue
            rows.append({"source": source, "country": country, "slice": name, "n": int(m.sum()),
                         "recall@1": round(float((rank[m] < 1).mean()), 3),
                         "recall@5": round(float((rank[m] < 5).mean()), 3),
                         "recall@10": round(float((rank[m] < 10).mean()), 3),
                         "true cos (median)": round(float(np.median(true_cos[m])), 3)})
        no_match = sub["kind"].to_numpy() == "no_match"
        if no_match.sum():
            rows.append({"source": source, "country": country, "slice": "no-match: best S1 cos (median)",
                         "n": int(no_match.sum()), "true cos (median)": round(float(np.median(best_cos[no_match])), 3)})
    return rows


CHECK = pd.DataFrame(check("s2", S2) + check("s3", S3))
CHECK.to_csv(OUT_DIR / "quick_check.csv", index=False)
CHECK

## A top-k candidate bucket for every S1: exact search and FAISS IVF-PQ

**Goal:** each S1 gets a bucket of `TOP_K` *potential* matches from S2/S3. What matters is that
the true matches are in the bucket (**bucket recall**); reproducing exact nearest-neighbour search
is not the aim. Per country (matches never cross countries), all sampled S2+S3 embeddings are
indexed and every S1 of that country retrieves its bucket.

| Part | Setting | Why |
|---|---|---|
| Mean-centering | subtract the mean S2/S3 vector, re-normalize | e5 vectors share a large common direction (almost all cosines are 0.85–1.0). Removing it spreads them out and raised exact true-match recall@50 from 93.0% to 95.3% (India, 10% sample) |
| IVF | `nlist` ≈ √N clusters; `nprobe` = `NPROBE_FRAC` (25%) of them | a query scans only the nearest clusters. On this data this step is where recall is lost (see below) |
| PQ | 48 sub-vectors × 8 bits = 48 bytes/vector | memory: 48 bytes instead of 1,536 (fp32) per vector |
| Exact re-ranking | top `REFINE_FACTOR`·k (10·k) PQ hits re-scored exactly | after re-ranking, PQ cost **no** recall in our tests (IVF-Flat gave identical recall) |

**Measured on the 10% sample** (India, true-match recall in a top-50 bucket): exact search 95.3%,
IVF scanning 25% of clusters 85.6%, 10% 74.8%, 5% 66.8%; HNSW (efSearch 256) 89.3%. Training the
clusters on S1+S2/S3 did not help. `BUCKET_METHOD` picks which buckets are saved as
`s1_buckets.parquet` and scored: `ivfpq` (default) computes both methods and compares them, `exact`
runs exact search only.

**Exact search on the GPUs:**
* Every GPU holds a copy of the country's S2+S3 vectors and takes its share of the S1 queries,
  in blocks sized to keep each score matrix under about 1 GB.
* Each block is scored against all vectors in one matrix product, in fp16 on tensor-core GPUs.
* The best k + 32 per S1 are then re-scored in fp32, so rounding cannot reorder the final top k.
* The original e5 cosine of every bucket pair is also computed on the GPU.
* Without a GPU, FAISS `IndexFlatIP` runs on the CPU and gives the same results.

At full scale (0.7–1.3M S1 against 3.8–4.7M S2+S3 per country), this is on the order of minutes
per country on two T4s. IVF-PQ stays on the CPU (`faiss-cpu`). It saves memory only without its
exact re-ranking, which here keeps a full copy of the vectors. Raise `NPROBE_FRAC` to trade IVF
speed for recall.

In [ ]:
import importlib.util
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor

if importlib.util.find_spec("faiss") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
import faiss

PQ_M, PQ_NBITS = 48, 8                                         # 384 / 48 = 8 dims per sub-vector
REFINE_FACTOR = 10                                             # exact re-ranking of the top 10·k PQ hits
METHODS = ["ivfpq", "exact"] if BUCKET_METHOD == "ivfpq" else ["exact"]
FAISS_DIR = OUT_DIR / "faiss"
FAISS_DIR.mkdir(exist_ok=True)


def centered(vecs, mu):
    """Subtract mu and re-normalize rows (float32, C-contiguous) for FAISS."""
    out = vecs.astype(np.float32) - mu
    out /= np.maximum(np.linalg.norm(out, axis=1, keepdims=True), 1e-12)
    return np.ascontiguousarray(out, dtype=np.float32)


def build_ivfpq(vecs):
    """IVF-PQ (inner product) wrapped in exact re-ranking. Returns (index, ivfpq, nlist)."""
    n, d = vecs.shape
    nlist = int(np.clip(np.sqrt(n), 16, max(16, n // 39)))
    quantizer = faiss.IndexFlatIP(d)
    ivfpq = faiss.IndexIVFPQ(quantizer, d, nlist, PQ_M, PQ_NBITS, faiss.METRIC_INNER_PRODUCT)
    ivfpq.train(vecs)
    index = faiss.IndexRefineFlat(ivfpq)        # keeps full vectors for exact re-ranking
    index.k_factor = REFINE_FACTOR
    index.add(vecs)                             # adds to both the PQ index and the exact store
    ivfpq.nprobe = max(1, round(NPROBE_FRAC * nlist))
    return index, ivfpq, nlist


SEARCH_DTYPE = torch.float16 if USE_FP16 else torch.float32    # first scoring pass (fp16 = tensor cores)
RESCORE_EXTRA = 32                                             # the top k + 32 of that pass are re-scored in fp32


def exact_search(vecs, queries, k):
    """Exact top-k by inner product.

    With GPUs: every GPU holds a copy of the S2/S3 vectors and takes its share of the S1 query blocks
    (one thread per GPU). A block is scored against all vectors in one matrix product, then its best
    k + RESCORE_EXTRA are re-scored in fp32 and the top k kept. Without a GPU: FAISS IndexFlatIP.
    """
    if not GPUS:
        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        return index.search(queries, k)
    shortlist = min(len(vecs), k + RESCORE_EXTRA)
    rows = int(np.clip(2.5e8 // len(vecs), 64, 4096))          # queries per block: score block <= ~1 GB
    blocks = [(s, min(s + rows, len(queries))) for s in range(0, len(queries), rows)]
    scores = np.empty((len(queries), k), np.float32)
    hits = np.empty((len(queries), k), np.int64)

    def run(device, my_blocks):
        with torch.cuda.device(device):
            base = torch.from_numpy(vecs).to(device, SEARCH_DTYPE)
            for s, e in my_blocks:
                q = torch.from_numpy(queries[s:e]).to(device)
                cand = torch.topk(q.to(SEARCH_DTYPE) @ base.T, shortlist, dim=1).indices
                rescored = torch.einsum("qd,qkd->qk", q, base[cand].float())
                top = torch.topk(rescored, k, dim=1)
                scores[s:e] = top.values.cpu().numpy()
                hits[s:e] = torch.gather(cand, 1, top.indices).cpu().numpy()
            del base
            torch.cuda.empty_cache()

    with ThreadPoolExecutor(len(GPUS)) as ex:
        list(ex.map(run, GPUS, [blocks[i::len(GPUS)] for i in range(len(GPUS))]))
    return scores, hits


def pair_cosine(a, a_rows, b, b_rows, chunk=250_000):
    """Dot product of a[a_rows[j]] and b[b_rows[j]] for every j (original e5 cosine), in chunks, on the GPU when there is one."""
    out = np.empty(len(a_rows), np.float32)
    if GPUS:
        with torch.cuda.device(GPUS[0]):
            ta, tb = torch.from_numpy(a).to(GPUS[0]), torch.from_numpy(b).to(GPUS[0])
            for s in range(0, len(a_rows), chunk):
                ia = torch.from_numpy(a_rows[s:s + chunk]).to(GPUS[0])
                ib = torch.from_numpy(b_rows[s:s + chunk]).to(GPUS[0])
                out[s:s + chunk] = (ta[ia].float() * tb[ib].float()).sum(dim=1).cpu().numpy()
            del ta, tb
            torch.cuda.empty_cache()
        return out
    for s in range(0, len(a_rows), chunk):
        out[s:s + chunk] = np.einsum("ij,ij->i", a[a_rows[s:s + chunk]].astype(np.float32),
                                     b[b_rows[s:s + chunk]].astype(np.float32))
    return out


def bucket_frame(hits, scores, q_rows, c_rows, country, method):
    """Long table of buckets: one row per (S1, candidate) with rank, centered score and original cosine."""
    k = hits.shape[1]
    valid = (hits >= 0).ravel()
    qi = np.repeat(np.arange(len(q_rows)), k)[valid]
    ci = hits.ravel()[valid]
    return pd.DataFrame({
        "s1_entity_id": S1["entity_id"].to_numpy()[q_rows][qi],
        "rank": np.tile(np.arange(k), len(q_rows))[valid],
        "candidate_entity_id": CANDIDATES["entity_id"].to_numpy()[c_rows][ci],
        "candidate_source": CANDIDATES["source"].to_numpy()[c_rows][ci],
        "score": scores.ravel()[valid].astype(np.float32),                          # centered cosine (retrieval order)
        "cosine": pair_cosine(EMB["s1"][q_rows], qi, CAND_EMB[c_rows], ci),        # original e5 cosine
        "country": country, "method": method})


CANDIDATES = pd.concat([S2.assign(source="S2"), S3.assign(source="S3")], ignore_index=True)
CAND_EMB = np.vstack([EMB["s2"], EMB["s3"]])                   # float16, row-aligned with CANDIDATES
frames = {m: [] for m in METHODS}
faiss_stats = []
for country in sorted(S1["country"].unique()):
    c_rows = np.flatnonzero((CANDIDATES["country"] == country).to_numpy())
    q_rows = np.flatnonzero((S1["country"] == country).to_numpy())
    if len(c_rows) < 1_000:
        print(f"{country}: only {len(c_rows)} S2/S3 records in the sample, no index built "
              "(France has no S2/S3 files in processed/)")
        continue
    mu = CAND_EMB[c_rows].astype(np.float32).mean(axis=0, keepdims=True)
    vecs, queries = centered(CAND_EMB[c_rows], mu), centered(EMB["s1"][q_rows], mu)
    np.save(FAISS_DIR / f"mean_{country}.npy", mu)             # needed to query with the same centering later
    k = min(TOP_K, len(c_rows))
    stats = {"country": country, "S2+S3 vectors": len(c_rows), "S1 queries": len(q_rows), "k": k}
    if "ivfpq" in METHODS:
        t0 = time.time()
        index, ivfpq, nlist = build_ivfpq(vecs)
        t_build = time.time() - t0
        t0 = time.time()
        scores, hits = index.search(queries, k)
        t_search = time.time() - t0
        frames["ivfpq"].append(bucket_frame(hits, scores, q_rows, c_rows, country, "ivfpq"))
        faiss.write_index(index, str(FAISS_DIR / f"index_{country}.faiss"))
        del index
        stats.update({"nlist": nlist, "nprobe": ivfpq.nprobe, "IVF-PQ build s": round(t_build, 1),
                      "IVF-PQ search s": round(t_search, 1)})
    t0 = time.time()
    e_scores, e_hits = exact_search(vecs, queries, k)
    stats.update({"exact search s": round(time.time() - t0, 1), "exact on": f"{len(GPUS)} GPU" if GPUS else "CPU"})
    frames["exact"].append(bucket_frame(e_hits, e_scores, q_rows, c_rows, country, "exact"))
    faiss_stats.append(stats)
    print(", ".join(f"{key} {val:,}" if isinstance(val, int) else f"{key} {val}" for key, val in stats.items()))

BUCKETS_ALL = {m: pd.concat(f, ignore_index=True) for m, f in frames.items()}
for m, b in BUCKETS_ALL.items():
    b.to_parquet(FAISS_DIR / f"s1_buckets_{m}.parquet", index=False)
BUCKETS = BUCKETS_ALL[BUCKET_METHOD]
BUCKETS.to_parquet(FAISS_DIR / "s1_buckets.parquet", index=False)
print(f"buckets saved; s1_buckets.parquet = {BUCKET_METHOD}")
pd.DataFrame(faiss_stats)

### How many true matches land in each bucket?
For S1 with at least one true match inside the sample, **bucket recall@k** is the share of those
matches found in the S1's top-k. It is averaged per S1 (like the challenge metric) and pooled over
all pairs, for IVF-PQ and exact search side by side.

In [ ]:
TRUE_OF = CANDIDATES[CANDIDATES["kind"] == "linked"].groupby("true_s1")["entity_id"].apply(set).to_dict()


def with_labels(buckets):
    """Add is_true (candidate is a true match of that S1) and is_owner (S1 is the candidate's best-scoring S1)."""
    b = buckets.copy()
    b["is_true"] = [c in TRUE_OF.get(s, ()) for s, c in zip(b["s1_entity_id"], b["candidate_entity_id"])]
    owner = (b.sort_values("cosine", ascending=False).drop_duplicates("candidate_entity_id")
             .set_index("candidate_entity_id")["s1_entity_id"])
    b["is_owner"] = b["candidate_entity_id"].map(owner).to_numpy() == b["s1_entity_id"].to_numpy()
    return b


LABELED = {m: with_labels(b) for m, b in BUCKETS_ALL.items()}
n_true_all = pd.Series({s: len(v) for s, v in TRUE_OF.items()}, name="n_true")
rows = []
for method, b in LABELED.items():
    for k in (1, 5, 10, 20, 50):
        if k > TOP_K:
            continue
        found = b[b["is_true"] & (b["rank"] < k)].groupby("s1_entity_id").size()
        d = n_true_all[n_true_all.index.isin(b["s1_entity_id"].unique())].to_frame()
        d["found"] = found.reindex(d.index).fillna(0)
        d["country"] = d.index.map(S1.set_index("entity_id")["country"])
        for country, g in d.groupby("country"):
            rows.append({"country": country, "k": k, "method": method, "S1 with matches": len(g),
                         "bucket recall (mean per S1)": round(float((g["found"] / g["n_true"]).mean()), 4),
                         "bucket recall (all pairs)": round(float(g["found"].sum() / g["n_true"].sum()), 4)})
BUCKET_RECALL = pd.DataFrame(rows)
BUCKET_RECALL.to_csv(FAISS_DIR / "bucket_recall.csv", index=False)
display(BUCKET_RECALL.pivot(index=["country", "k"], columns="method",
                            values=["bucket recall (mean per S1)", "bucket recall (all pairs)"]))

top1 = LABELED[BUCKET_METHOD][LABELED[BUCKET_METHOD]["rank"] == 0].merge(
    S1[["entity_id", "kind"]], left_on="s1_entity_id", right_on="entity_id")
print("top-1 original cosine by S1 kind (can a score alone spot singletons?):")
top1.groupby(["country", "kind"])["cosine"].describe(percentiles=[0.1, 0.5, 0.9]).round(3)

### Challenge metric (macro F0.5) on the buckets
Scored over the sampled **labeled** S1: true singletons, plus S1 with at least one true match
inside the sample. "True matches" means the sampled S2/S3 records linked to that S1, since
unsampled ones cannot be retrieved. Rows:
* **Ceiling:** a perfect matcher that keeps exactly the true matches in the top-k bucket. This is
  the best score the buckets allow, shown for IVF-PQ and exact search.
* **Predict nothing / whole top-k:** naive baselines.
* **Cosine ≥ t:** keep bucket entries whose original e5 cosine is above a cutoff. With "one owner",
  each S2/S3 is only kept for the S1 that scores it highest, because every S2/S3 has at most one
  true S1. The cutoff is tuned on this same sample, so treat it as optimistic.

In [ ]:
EVAL_IDS = S1.loc[(S1["split"] == "train") & ((S1["kind"] == "singleton") | S1["entity_id"].isin(TRUE_OF)),
                  "entity_id"]
N_TRUE = pd.Series({s: len(TRUE_OF.get(s, ())) for s in EVAL_IDS}, name="n_true")
EVAL_COUNTRY = N_TRUE.index.map(S1.set_index("entity_id")["country"]).to_numpy()
EVAL_POS = pd.Series(np.arange(len(N_TRUE)), index=N_TRUE.index)   # S1 id -> position in N_TRUE


def score(b, mask):
    """Challenge macro F0.5 when the rows of bucket table b selected by mask are the predictions (vectorized)."""
    pos = b["s1_pos"].to_numpy()[mask]
    n_pred = np.bincount(pos, minlength=len(N_TRUE))
    tp = np.bincount(pos, weights=b["is_true"].to_numpy()[mask].astype(float), minlength=len(N_TRUE))
    n_true = N_TRUE.to_numpy()
    with np.errstate(divide="ignore", invalid="ignore"):
        p, r = tp / n_pred, tp / n_true
        f = np.where(tp > 0, 1.25 * p * r / (0.25 * p + r), 0.0)
    f = np.where(n_true == 0, (n_pred == 0).astype(float), f)
    out = {"macro F0.5": f.mean(), "S1 with matches": f[n_true > 0].mean(), "singletons": f[n_true == 0].mean()}
    out.update({f"macro F0.5 {c}": f[EVAL_COUNTRY == c].mean() for c in sorted(set(EVAL_COUNTRY))})
    return out


EVAL = {}
for method, b in LABELED.items():
    EVAL[method] = b[b["s1_entity_id"].isin(N_TRUE.index)].copy()
    EVAL[method]["s1_pos"] = EVAL[method]["s1_entity_id"].map(EVAL_POS).to_numpy()
B = EVAL[BUCKET_METHOD]
rows = [{"method": "predict nothing", "buckets": "-", "k": 0, "cosine ≥": None, **score(B, np.zeros(len(B), bool))}]
for method, b in EVAL.items():
    for k in (1, 5, 10, 20, 50):
        if k <= TOP_K:
            rows.append({"method": "ceiling: perfect matcher on the top-k bucket", "buckets": method, "k": k,
                         "cosine ≥": None, **score(b, (b["is_true"] & (b["rank"] < k)).to_numpy())})
for k in (1, 5):
    rows.append({"method": "predict the whole top-k bucket", "buckets": BUCKET_METHOD, "k": k, "cosine ≥": None,
                 **score(B, (B["rank"] < k).to_numpy())})
THRESHOLDS = np.round(np.arange(0.80, 0.995, 0.005), 3)
cos = B["cosine"].to_numpy()
BEST_T = {}
for label, extra in [("cosine ≥ t", np.ones(len(B), bool)),
                     ("cosine ≥ t, one owner per S2/S3", B["is_owner"].to_numpy())]:
    sweep = [(t, score(B, (cos >= t) & extra)) for t in THRESHOLDS]
    t_best, best = max(sweep, key=lambda x: x[1]["macro F0.5"])
    BEST_T[label] = t_best
    rows.append({"method": label + " (t tuned on this sample)", "buckets": BUCKET_METHOD, "k": TOP_K,
                 "cosine ≥": t_best, **best})
METRIC = pd.DataFrame(rows)
METRIC.to_csv(FAISS_DIR / "metric_f05.csv", index=False)

# cross-check the vectorized score against the reference f05() for the best one-owner cutoff
t = BEST_T["cosine ≥ t, one owner per S2/S3"]
kept = B[(cos >= t) & B["is_owner"].to_numpy()]
pred = kept.groupby("s1_entity_id")["candidate_entity_id"].apply(list).to_dict()
reference = macro_f05(pred, TRUE_OF, N_TRUE.index)
assert abs(reference - METRIC.iloc[-1]["macro F0.5"]) < 1e-9, (reference, METRIC.iloc[-1]["macro F0.5"])
print(f"{len(N_TRUE):,} labeled S1 scored ({int((N_TRUE == 0).sum()):,} singletons); "
      f"vectorized score matches the reference metric")
METRIC.round(4)

### Look at the bucket of a particular S1
`show_bucket(s1_entity_id, k)` prints the S1 record and its top-k candidates, marking true
matches. With no argument it picks a random S1 that has matches in the sample.

In [ ]:
S1_BY_ID = S1.set_index("entity_id")
CAND_BY_ID = CANDIDATES.set_index("entity_id")


def show_bucket(s1_entity_id=None, k=10, seed=None):
    """Print one S1 record and its top-k FAISS bucket; ✓ marks a true match."""
    if s1_entity_id is None:
        with_matches = [s for s in BUCKETS["s1_entity_id"].unique() if s in TRUE_OF]
        s1_entity_id = np.random.default_rng(seed).choice(with_matches)
    s1 = S1_BY_ID.loc[s1_entity_id]
    truth = TRUE_OF.get(s1_entity_id, set())
    print(f"S1 {s1_entity_id} [{s1['country']}, {s1['kind']}]: {s1['business_name']} | {s1['business_address']}")
    print(f"true matches in the sample: {len(truth)}; found in top-{k}: "
          f"{len(truth & set(BUCKETS[(BUCKETS['s1_entity_id'] == s1_entity_id) & (BUCKETS['rank'] < k)]['candidate_entity_id']))}")
    b = BUCKETS[(BUCKETS["s1_entity_id"] == s1_entity_id) & (BUCKETS["rank"] < k)].sort_values("rank")
    out = b.assign(match=b["candidate_entity_id"].map(lambda e: "✓" if e in truth else ""),
                   name=b["candidate_entity_id"].map(CAND_BY_ID["business_name"]),
                   address=b["candidate_entity_id"].map(CAND_BY_ID["business_address"]),
                   kind=b["candidate_entity_id"].map(CAND_BY_ID["kind"]))
    # rank follows the centered retrieval score; cosine is the original e5 cosine
    return out[["rank", "score", "cosine", "match", "candidate_entity_id", "candidate_source", "kind", "name", "address"]]


show_bucket(seed=0)